# Delay Prediction Exercises

This exercise uses package [delivery](../dataset/syndelay_v1.csv) dataset. The data is downloaded from [supply chain data hub](https://supplychaindatahub.org/datasets/syndelay/). 

**Data Set Information:**

The dataset was generated synthetically through generative model trained on real-world data.

**Attribute Information:**

The data comprises 40 attributes and 1 target column namely `label`. Out of 40 attributes, 26 of them contain numeric values while the rest contain text value. The target `label` column has only three values (i.e., 0, 1, 2).
The columns of `order_date` and `shipping_date` are both in decimal values (float). Hence, both are required to be converted as `date` datatype.

## Exercise goal

The goal of this exercise is to build predictive models that can predict the delivery delay. Since the target `label` has three different class (0, 1, 2), we can define the problem as classification. 

To complete this exercise, please refer to [classification](classification.ipynb) notebook.

In [1]:
# load libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

In [3]:
delivery = pd.read_csv('../dataset/syndelay_v1.csv')
delivery.head()

,payment_type,profit_per_order,sales_per_customer,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,...,order_region,order_state,order_status,product_card_id,product_category_id,product_name,product_price,shipping_date,shipping_mode,label
0,PAYMENT,-32.924488,278.95000,38,Kids' Golf Clubs,Caguas,Puerto Rico,12446.5625,Corporate,PR,...,Caribbean,Martinique,PENDING_PAYMENT,858,38,GolfBuddy VT3 GPS Watch,129.99,42177.500,Second Class,2
1,DEBIT,107.874500,263.98000,17,Cleats,Caguas,Puerto Rico,7782.0170,Corporate,PR,...,East Africa,Copperbelt,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,42502.390,Same Day,1
2,PAYMENT,35.770718,109.65013,17,Cleats,Caguas,Puerto Rico,7378.1113,Consumer,PR,...,West Asia,Ankara,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,42951.266,Standard Class,0
3,PAYMENT,43.587560,113.09000,18,Men's Footwear,Caguas,Puerto Rico,1448.6765,Consumer,PR,...,Central America,Francisco Morazan,PENDING_PAYMENT,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,42181.900,Second Class,2
4,PAYMENT,49.804802,191.98090,9,Cardio Equipment,Madison,EE. UU.,5123.5254,Corporate,WI,...,Central America,Leon,PENDING_PAYMENT,191,9,Nike Men's Free 5.0+ Running Shoe,99.99,42632.820,Standard Class,1


In [4]:
# convert date datatype
delivery['order_date'] = pd.to_timedelta(delivery['order_date'], unit='D') + pd.to_datetime("1899-12-30")
delivery['shipping_date'] = pd.to_timedelta(delivery['shipping_date'], unit='D') + pd.to_datetime("1899-12-30")

## Exercise 1
Try to get basic information of the dataset and statistical summary of numeric columns only

<details>
  <summary>Click for answer</summary>
    
  ```python
  # display basic info
  delivery.info()
  
  # statistical summary with 2 decimal places
  delivery.select_dtypes(include='number').describe().T.round(2)

## Exercise 2
Check the proportion of target feature - `label` by plotting its count

<details>
  <summary>Click for answer</summary>
    
  ```python
  sns.countplot(data=delivery, x='label')

## Exercise 3

Make a box plot visualizing the distribution of sales by order status relative to their shipping mode

<details>
  <summary>Click for answer</summary>
    
  ```python
  plt.figure(figsize=(10, 5))
  sns.boxplot(data=delivery, x='sales', y='order_status', hue='shipping_mode', gap=.2, fliersize=3, palette='coolwarm')

## Exercise 4

Make a scatter plot visualizing order size per market area relative to their average sales and average profit. Use `customer_segment` as color to map different segments. Set `market` as grid columns to put a plot for each area. 

To make this plot, you first need a pivot table that groups the data by `market`, `customer_segment`, and `label`. This table aggregates counts of `order_id` and average of `sales` and `order_profit_per_order`.

<details>
  <summary>Click for answer</summary>
    
  ```python
  # create pivot table
  segment = delivery.groupby(['market', 'customer_segment', 'label'],as_index= False).agg(
    order_counts = ('order_id', 'count'), 
    sales_mean = ('sales', 'mean'), 
    profit_mean = ('order_profit_per_order', 'mean'), 
  )
  segment

<details>
  <summary>Click for answer</summary>
    
  ```python
  # create scatter plot
  # you may change hue option to another categorical value such as label
  sns.relplot(data=segment, x='sales_mean', y='profit_mean', size='order_counts', col='market', hue='customer_segment', col_wrap=3, sizes=(50, 250))

## Exercise 5
1. Calculate correlation matrix of numerical attributes.
2. Plot correlation matrix.

<details>
  <summary>Click for answer</summary>
    
  ```python
  # calculate correlation matrix
  corr = delivery.select_dtypes(include='number').corr()
  
  # plot correlation in heatmap plot
  plt.figure(figsize=(8,6))
  sns.heatmap(corr, cmap='vlag', linewidths=.5)

## Exercise 6
1. Use all numerical attributes only as predictor
2. Keep `label` column as target
3. Split data into train and test set each with proportion of 80% and 20%, respectively.

<details>
  <summary>Click for answer</summary>
    
  ```python
    from sklearn.model_selection import train_test_split
    
    X = delivery.select_dtypes(include='number').drop('label', axis=1) # use all features for predictor, except label
    y = delivery['label'] # target to predict
    
    # split dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    y_train.value_counts()

## Exercise 7
1. Get all numeric features, except `label` column
2. Create a pre-processor using column transformer to scaling numeric features with standard scaler
3. Refer to this [link](https://scikit-learn.org/stable/modules/preprocessing.html) for further information.

<details>
  <summary>Click for answer</summary>
    
  ```python
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    # get numeric features
    num_features = delivery.select_dtypes(include='number').drop('label', axis=1).columns.tolist()

    # create pre-processor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_features)
        ]
    )

## Exercise 8
1. Make `KNN`, `SVM`, and `RandomForest` classifiers
2. Make a pipeline to streamline preprocessor and classification steps
3. Train the pipeline using train dataset
4. Refer to this [link](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) for more information about **Random Forest**
5. Refer to this [link](https://scikit-learn.org/stable/modules/compose.html) for more information about **pipeline** implementation

<details>
  <summary>Click for answer</summary>
    
  Uppss.. no quick answer found. Try to figure it out.

## Exercise 9
1. Predict the models using test set.
2. Evaluate models' performance by print out classification report and plot the confusion matrix
3. Make your own analysis on the results

<details>
  <summary>Click for answer</summary>
    
  You click it again! But you're not lucky this time.